In [1]:
import csv
from collections import defaultdict
from datetime import datetime
import json
from pathlib import Path
import re
import time
from traceback import format_exc
from typing import (
    Any,
    Dict,
    List,
    Optional
)

In [2]:
import boto3
import fitz
from trp.trp2_analyzeid import TAnalyzeIdDocument, TAnalyzeIdDocumentSchema
from trp import Document

In [3]:
import nconfig as nb_config
from config import config as server_config
from backend.common.logging import logging

In [4]:
logger = logging.getLogger("sp2_text_parser")

In [5]:
FILENAME = '/home/lap-49/Documents/ot-report/assets/inputs/images/Sensory-image-Profile-2-Summary-Report_70247631_1751134355067.pdf'


In [6]:

def initialize_aws_textract_client(
        aws_access_key_id: Optional[str] = None,
        aws_secret_access_key: Optional[str] = None,
        region_name: str = 'us-east-1'
    ):
    """
    Initialize the Textract OCR analyzer
    
    Args:
        aws_access_key_id: AWS access key ID (optional, can use env vars)
        aws_secret_access_key: AWS secret access key (optional, can use env vars)
        region_name: AWS region name
        output_dir: Directory to store output files
    """
    region_name = region_name
    
    # Initialize Textract client
    try:
        session_kwargs = {'region_name': region_name}
        if aws_access_key_id and aws_secret_access_key:
            session_kwargs.update({
                'aws_access_key_id': aws_access_key_id,
                'aws_secret_access_key': aws_secret_access_key
            })
        
        textract_client = boto3.client('textract', **session_kwargs)
        logger.info(f"Initialized Textract client for region: {region_name}")
        return textract_client
    except Exception as e:
        logger.error(f"Failed to initialize Textract client: {e}")
        raise


In [7]:
analyzer = initialize_aws_textract_client(
    aws_access_key_id=server_config.AMAZON_ACCESS_KEY_ID,
    aws_secret_access_key=server_config.AMAZON_SECRET_ACCESS_KEY,
    region_name=server_config.AMAZON_REGION,
)

2248436423.py - 2025-07-25 11:18:32,658 - sp2_text_parser - INFO - Initialized Textract client for region: us-east-1


In [8]:

def extract_pages_as_bytes(pdf_path: str) -> List[bytes]:
    """
    Extract all pages from PDF as bytes
    
    Args:
        pdf_path: Path to the PDF file
        
    Returns:
        List of page bytes
    """
    try:
        logger.info(f"📄 Extracting pages from PDF: {pdf_path}")
        
        # Open PDF document
        doc = fitz.open(pdf_path)
        page_bytes_list = []
        
        for page_num in range(len(doc)):
            logger.info(f"🔄 Processing page {page_num + 1}/{len(doc)}")
            
            # Get page
            page = doc[page_num]
            
            # Convert page to image (PNG format)
            pix = page.get_pixmap(dpi=300)  # High DPI for better OCR
            img_data = pix.tobytes("png")
            
            page_bytes_list.append(img_data)
        
        doc.close()
        logger.info(f"✅ Successfully extracted {len(page_bytes_list)} pages as bytes")
        return page_bytes_list

    except Exception as e:
        logger.error(f"Failed to extract pages from PDF: {e}")
        raise
            

In [9]:
full_response = {}

In [10]:

def analyze_document(pdf_path: str) -> Dict[str, Any]:
    """
    Analyze a PDF document to extract tables using analyze_document API
    
    Args:
        pdf_path: Path to the PDF file
        
    Returns:
        Dictionary containing all page analysis results
    """
    try:
        logger.info(f"🔍 Starting OCR table analysis of document: {pdf_path}")
        
        # Extract pages as bytes
        page_bytes_list = extract_pages_as_bytes(pdf_path)
        
        # Analyze each page
        all_responses = []
        
        for page_num, page_bytes in enumerate(page_bytes_list):
            logger.info(f"📊 Analyzing page {page_num + 1}/{len(page_bytes_list)}")
            
            try:
                # Call Textract analyze_document for this page
                response = analyzer.analyze_document(
                    Document={'Bytes': page_bytes},
                    FeatureTypes=['TABLES', 'FORMS']
                )

                with open(f"{OUTPUT_DIR}/aws_sp2_page_{page_num}.json", 'w+') as f:
                    f.write(json.dumps(response, indent=4))

                all_responses.append(response)
                
                # Add small delay to avoid rate limiting
                time.sleep(0.1)
                
            except Exception as e:
                print(format_exc())
                logger.error(f"Failed to analyze page {page_num + 1}: {e}")
                # Continue with next page
        
       
        logger.info(f"✅ Successfully analyzed document: {pdf_path}")
        return all_responses
        
    except Exception as e:
        logger.error(f"Error analyzing document {pdf_path}: {e}")
        raise


In [11]:
all_responses = analyze_document(FILENAME)

2018989099.py - 2025-07-25 11:18:32,706 - sp2_text_parser - INFO - 🔍 Starting OCR table analysis of document: /home/lap-49/Documents/ot-report/assets/inputs/images/Sensory-image-Profile-2-Summary-Report_70247631_1751134355067.pdf
382849967.py - 2025-07-25 11:18:32,707 - sp2_text_parser - INFO - 📄 Extracting pages from PDF: /home/lap-49/Documents/ot-report/assets/inputs/images/Sensory-image-Profile-2-Summary-Report_70247631_1751134355067.pdf
382849967.py - 2025-07-25 11:18:32,722 - sp2_text_parser - INFO - 🔄 Processing page 1/14
382849967.py - 2025-07-25 11:18:34,207 - sp2_text_parser - INFO - 🔄 Processing page 2/14
382849967.py - 2025-07-25 11:18:35,660 - sp2_text_parser - INFO - 🔄 Processing page 3/14
382849967.py - 2025-07-25 11:18:37,090 - sp2_text_parser - INFO - 🔄 Processing page 4/14
382849967.py - 2025-07-25 11:18:38,404 - sp2_text_parser - INFO - 🔄 Processing page 5/14
382849967.py - 2025-07-25 11:18:39,774 - sp2_text_parser - INFO - 🔄 Processing page 6/14
382849967.py - 2025-0

In [12]:

def merge_textract_json_files(responses: list[bytes]):
    """
    Merges multiple AWS Textract JSON response files into a single file.

    Args:
        file_paths (list): A list of paths to the JSON files to merge.
        output_file (str): The path to save the merged JSON file.
    """
    merged_data = responses[0]

    # Overwrite page count based on files merged
    merged_data["DocumentMetadata"]["Pages"] = len(responses)

    # Append blocks from remaining files
    for idx, data in enumerate(responses[1:], start=2):
        for block in data.get('Blocks', []):
            block['Page'] = idx
        merged_data['Blocks'].extend(data.get('Blocks', []))
        
    print(f"Successfully merged {len(responses)}")
    return merged_data


In [13]:
response_merged = merge_textract_json_files(all_responses)

Successfully merged 14


In [14]:
document = Document(response_merged)

In [34]:
import re
from collections import defaultdict

def extract_behavioral_items(document):
    examples = defaultdict(list)

    for page in document.pages:
        for table in page.tables:
            for row in table.rows:
                cells = [cell.text.strip() for cell in row.cells]
                if not cells or len(cells) < 3:
                    continue

                code = cells[0]
                text = cells[2].strip('"').rstrip("* ")
                if not re.match(r'^(SK|AV|SN|ET|EM|VO)$', code):
                    continue

                # Look for any SELECTED in other cells (e.g., responses)
                if any("SELECTED" in cell for cell in cells[3:]):
                    if code == "AV":
                        examples["Avoiding"].append(text)
                    elif code == "SK":
                        examples["Seeking"].append(text)
                    elif code == "SN":
                        examples["Sensitivity"].append(text)
                    elif code == "ET":
                        examples["Touch"].append(text)
                    elif code == "VO":
                        examples["Visual"].append(text)

    return dict(examples)
